# 기능 3 — 상태 변화 분석 (Google Colab T4)

**이 노트북은 실제 카카오톡 단체 채팅 로그를 입력으로 돌리기 위해 작성되었습니다.**
익명화·PII 마스킹을 마친 실데이터 두 개(`chat_log1_raw.json`, `chat_log2_raw.json`)를
한 번의 **Run all** 로 **각각 순서대로** 돌리고, 결과를 입력 로그별 폴더
(`outputs/<입력명>_<실행시각>/`)에 따로 저장합니다.

**처리 범위 (로그 1개당)**
- 기능 1·2 모델 추론으로 감정/공격성 라벨링 → 라벨 검수 파일 저장
- 오늘 vs 직전 7일 비교, 3대 지표 연산
- 종합 상태(🟢🟡🟠⚪) 판정 + 자연어 해석 → `daily_report.json`

## CELL 1 — 환경 설정 (T4 런타임)

런타임 → **T4 GPU** 선택 후 실행. 기능 3 연산은 CPU만으로도 동작하지만, 기능 1·2 추론에 GPU를 씁니다.

In [ ]:
# Colab 기본 패키지 (기능 3은 표준 라이브러리만 사용)
import json
import sys
import subprocess
from datetime import date
from pathlib import Path

# 저장소 자동 clone — Run all만으로 기능3 코드(state_change_analysis.py)와
# 입력(chat_log1_raw.json, chat_log2_raw.json)을 확보한다. (이미 있으면 재사용)
REPO_URL = "https://github.com/SARA-MAYO/ChaeOn-AIProgramming.git"
REPO_DIR = Path("/content/ChaeOn-AIProgramming")
if not REPO_DIR.exists():
    print("저장소 clone 중...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("이미 clone된 저장소를 사용합니다.")

# 작업 디렉터리 = 기능3 폴더 (py 파일·입력 json 위치)
WORK_DIR = REPO_DIR / "feature3"
if not WORK_DIR.exists():
    WORK_DIR = Path(".")  # 폴백: 노트북과 같은 폴더에 파일이 있을 때

sys.path.insert(0, str(WORK_DIR))
print("WORK_DIR:", WORK_DIR.resolve())

In [2]:
import os
from google.colab import drive
import torch, joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── 산출물은 기능별 폴더에 저장됨 (기능1·2 노트북이 MyDrive/feature1, MyDrive/feature2 에 저장) ──
DRIVE = '/content/drive/MyDrive'
F1_DIR = os.path.join(DRIVE, 'feature1')   # 기능1 산출물 폴더
F2_DIR = os.path.join(DRIVE, 'feature2')   # 기능2 산출물 폴더

def _has_weights(d):
    return any(os.path.exists(os.path.join(d, w)) for w in ['model.safetensors', 'pytorch_model.bin'])

problems = []
# 모델 폴더: 폴더 존재 + 가중치 파일 존재까지 확인
for path, desc in {
    os.path.join(F1_DIR, 'chaeon_feature1_checkpoint'): '기능1 KcELECTRA 모델  → 기능1 노트북 Run all 필요',
    os.path.join(F2_DIR, 'chaeon_feature2_model'):      '기능2 KoELECTRA 모델  → 기능2 노트북 Run all 필요',
}.items():
    if not os.path.exists(path):
        problems.append(f"  - {path} 폴더 없음            ({desc})")
    elif not _has_weights(path):
        problems.append(f"  - {path} 폴더는 있으나 가중치(model.safetensors) 없음  ({desc})")
# pkl 파일 (기능1)
for path, desc in {
    os.path.join(F1_DIR, 'svm_model.pkl'):  '기능1 SVM 모델          → 기능1 노트북 Run all 필요',
    os.path.join(F1_DIR, 'vectorizer.pkl'): '기능1 TF-IDF 벡터라이저  → 기능1 노트북 Run all 필요',
}.items():
    if not os.path.exists(path):
        problems.append(f"  - {path} 없음            ({desc})")

if problems:
    raise FileNotFoundError(
        "기능3 실행에 필요한 파일이 Google Drive(MyDrive/feature1, MyDrive/feature2)에 없거나 불완전합니다:\n"
        + "\n".join(problems)
        + "\n\n→ 부족한 기능의 노트북을 먼저 'Run all' 하면 해당 기능 폴더에 자동 저장됩니다."
    )
print("필수 모델 파일(가중치 포함) 확인 완료")

# 기능1 로드 (MyDrive/feature1)
f1_ckpt = os.path.join(F1_DIR, 'chaeon_feature1_checkpoint')
f1_model = AutoModelForSequenceClassification.from_pretrained(f1_ckpt).to(device)
f1_tokenizer = AutoTokenizer.from_pretrained(f1_ckpt)
f1_svm = joblib.load(os.path.join(F1_DIR, 'svm_model.pkl'))
f1_vec = joblib.load(os.path.join(F1_DIR, 'vectorizer.pkl'))

# 기능2 로드 (MyDrive/feature2)
f2_ckpt = os.path.join(F2_DIR, 'chaeon_feature2_model')
f2_model = AutoModelForSequenceClassification.from_pretrained(f2_ckpt).to(device)
f2_tokenizer = AutoTokenizer.from_pretrained(f2_ckpt)

print("기능1·2 모델 로드 완료")

Mounted at /content/drive
필수 모델 파일(가중치 포함) 확인 완료


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

기능1·2 모델 로드 완료


## CELL 2 — 기능 1·2 모델 추론 인터페이스

앞 셀에서 **Drive로부터 자동 로드한** 기능 1·2 모델(`f1_*`, `f2_*`)을 호출해 원문 텍스트를 라벨링합니다.
별도 수정 없이 그대로 실행하면 됩니다.

In [3]:
import torch
import torch.nn.functional as F

# 기능1·2 추론 래퍼 — 모델은 CELL3에서 f1_*/f2_* 변수로 미리 로드해 둔다.

def run_feature1_emotion(text: str) -> str:
    # f1_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 모델 (epoch 2 체크포인트)
    # f1_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KcELECTRA 토크나이저
    # f1_svm       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 SVM 모델 (svm_model.pkl)
    # f1_vec       : 새 셀(Cell 1-A)에서 Drive로부터 로드한 TF-IDF 벡터라이저 (vectorizer.pkl)
    # device       : 새 셀(Cell 1-A)에서 설정한 'cuda' 또는 'cpu'
    f1_model.eval()
    inputs = f1_tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=128
    ).to(device)
    with torch.no_grad():
        kc_probs = F.softmax(f1_model(**inputs).logits, dim=-1).cpu().numpy()[0]
    svm_probs = f1_svm.predict_proba(f1_vec.transform([text]))[0]
    prob_pos = (kc_probs[1] * 0.7) + (svm_probs[1] * 0.3)  # KcELECTRA 7 : SVM 3 앙상블
    prob_neg = (kc_probs[0] * 0.7) + (svm_probs[0] * 0.3)
    return "negative" if prob_pos <= prob_neg else "positive"
def run_feature2_aggression(text: str) -> int:
    # f2_model     : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 모델 (checkpoint-5199)
    # f2_tokenizer : 새 셀(Cell 1-A)에서 Drive로부터 로드한 KoELECTRA-small 토크나이저
    f2_model.eval()
    inputs = f2_tokenizer(
        text, return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = f2_model(**inputs)
        pred_label = torch.argmax(outputs.logits, dim=-1).item()  # 0=비공격, 1=약한공격, 2=강한공격
    return int(pred_label)
def label_messages(raw_messages: list[dict]) -> list[dict]:
    # run_feature1_emotion, run_feature2_aggression 모두 위 f1_*/f2_* 변수에 의존
    # → 반드시 Cell 1-A(모델 로드 셀) 실행 후에 이 셀을 실행해야 함
    labeled = []
    for msg in raw_messages:
        text = msg["text"]
        labeled.append({
            "message_id":       msg["message_id"],
            "sender_id":        msg["sender_id"],
            "timestamp":        msg["timestamp"],
            "emotion_label":    run_feature1_emotion(text),    # 'negative' 또는 'positive'
            "aggression_label": run_feature2_aggression(text), # 0, 1, 2 정수
        })
    return labeled

## CELL 3 — 실데이터 채팅 로그 실행 (chat_log1 → chat_log2)

실제 카카오톡 로그 두 개를 **각각** 기능 1·2 모델로 라벨링한 뒤 기능 3 분석을 수행합니다.
아래 `process_input()` 한 번 호출 = 로그 1개 전체 처리(라벨링 → 검수 저장 → 날짜별 분석
→ 리포트 저장 → 시각화 → 자체 점검). 그 다음 셀에서 `chat_log1`·`chat_log2`를 반복 실행합니다.

> 산출물은 입력 로그별로 `outputs/chat_log1_raw_<시각>/`, `outputs/chat_log2_raw_<시각>/` 처럼
> **폴더가 분리**되어 서로 덮어쓰지 않습니다. (저장소 원본 입력 파일도 보존)

In [ ]:
# ============================================================================
# 실데이터 채팅 로그 1개를 기능 1·2·3 전체 파이프라인에 통과시키는 함수.
#   load → 기능1·2 라벨링 → 검수 저장 → 기능3 날짜별 분석 → 리포트 저장 → 시각화 → 자체 점검
#   산출물은 입력 로그별 outputs/<입력명>_<실행시각>/ 폴더에 분리 저장 (Drive 백업 포함).
# ============================================================================
import json, csv, random, shutil
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
from IPython.display import display, Markdown

from state_change_analysis import (
    MIN_MSG_COUNT, MIN_BASELINE_MSG_COUNT,
    run, run_daily, group_messages_by_sender, split_time_window,
)

_KST = timezone(timedelta(hours=9))


def _render_report(daily_reports, show_all_days=False, hide_empty_rows=True):
    """전체 요약 표(날짜×사용자) + 개인별 상세 멘트 시각화."""
    def _emoji(state):
        return state.split()[0] if state else "⚪"
    senders = sorted({r["sender_id"] for r in daily_reports})
    dates   = sorted({r["date"] for r in daily_reports})

    display(Markdown("### 전체 요약 — 날짜별 사용자 상태"))
    grid = {(r["sender_id"], r["date"]): _emoji(r["overall"]["state"]) for r in daily_reports}
    EMPTY_MARKS = {"⚪", "·"}
    if hide_empty_rows:
        table_dates = [d for d in dates
                       if not all(grid.get((s, d), "·") in EMPTY_MARKS for s in senders)]
    else:
        table_dates = dates
    df = pd.DataFrame(
        [[grid.get((s, d), "·") for s in senders] for d in table_dates],
        index=table_dates, columns=senders,
    )
    df.index.name = "날짜"
    COLOR = {"\U0001f7e2": "#d7f5dd", "\U0001f7e1": "#fff3cd",
             "\U0001f7e0": "#ffd9b3", "⚪": "#eeeeee", "·": "#ffffff"}
    def _cell_style(v):
        return f"background-color:{COLOR.get(v, '#ffffff')}; text-align:center; font-size:18px;"
    styler = df.style
    try:
        styler = styler.map(_cell_style)
    except AttributeError:
        styler = styler.applymap(_cell_style)
    display(styler)
    display(Markdown("\U0001f7e2 평소와 비슷 · \U0001f7e1 조금 다름 · \U0001f7e0 꽤 다름 · ⚪ 데이터 부족 · `·` 메시지 없음"))

    display(Markdown("### 개인별 상세"))
    for s in senders:
        rows = [r for r in daily_reports if r["sender_id"] == s]
        if not show_all_days:
            rows = [r for r in rows if r["data_sufficient"]]
        display(Markdown(f"#### \U0001f9d1 {s}"))
        if not rows:
            print("   (분석 가능한 날이 없습니다 — 메시지 10건 이상인 날 필요)")
            continue
        for r in rows:
            print(f"\n▸ {r['date']}   {r['overall']['state']}")
            if r["data_sufficient"]:
                m = r["metrics"]
                print(f"    부정 {m['negative']['change']:+.1f}%p({m['negative']['state']})   "
                      f"공격 {m['aggressive']['change']:+.1f}%p({m['aggressive']['state']})   "
                      f"참여 {m['participation']['change']:+.1f}%({m['participation']['state']})")
            print(f"    \U0001f4ac {r['overall']['text_interpretation']}")


def process_input(input_name):
    """실데이터 채팅 로그 1개를 전체 파이프라인에 통과시키고 결과를 폴더에 저장."""
    in_path = WORK_DIR / input_name
    if not in_path.exists():
        print(f"⏭️  {input_name} 없음 — 건너뜀")
        return
    display(Markdown(f"# ====== 입력: {input_name} ======"))

    with open(in_path, "r", encoding="utf-8") as f:
        RAW_CHAT_MESSAGES = json.load(f)
    print(f"✅ {input_name} 로드 완료: 총 {len(RAW_CHAT_MESSAGES)}건")

    # ── 이번 실행 산출물 폴더 (입력 로그별·실행시각별 분리) ──
    RUN_STAMP = datetime.now(_KST).strftime("%Y%m%d_%H%M%S")
    INPUT_STEM = Path(input_name).stem
    RUN_DIR = WORK_DIR / "outputs" / f"{INPUT_STEM}_{RUN_STAMP}"
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"\U0001f4c1 산출물 폴더: {RUN_DIR}")

    DRIVE_RUN_DIR = Path(f"/content/drive/MyDrive/feature3_outputs/{INPUT_STEM}_{RUN_STAMP}")
    try:
        DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    except Exception:
        DRIVE_RUN_DIR = None

    REFERENCE_DATE = None  # state_change_analysis 내부에서 데이터 최신 날짜 기준 자동 보정

    # === 기능 1·2 라벨링 ===
    if RAW_CHAT_MESSAGES and "emotion_label" in RAW_CHAT_MESSAGES[0]:
        print("\U0001f4a1 입력에 이미 라벨이 있어 모델 추론을 생략합니다.")
        labeled_messages = RAW_CHAT_MESSAGES
    elif RAW_CHAT_MESSAGES:
        print("\U0001f4a1 원문만 존재 → 기능 1·2 모델로 라벨링 시작...")
        labeled_messages = label_messages(RAW_CHAT_MESSAGES)
    else:
        labeled_messages = []
    if not labeled_messages:
        print("⚠️ 메시지가 없습니다 — 건너뜀")
        return
    print(f"라벨링 완료: {len(labeled_messages)}건 | 샘플: {labeled_messages[0]}")

    # === 기능1·2 실제 출력(labeled.json) 저장 (모델 추론한 경우만) ===
    made_by_model = bool(RAW_CHAT_MESSAGES) and "emotion_label" not in RAW_CHAT_MESSAGES[0]
    if made_by_model:
        out_path = RUN_DIR / "labeled.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(labeled_messages, f, ensure_ascii=False, indent=2)
        if DRIVE_RUN_DIR:
            shutil.copy(str(out_path), str(DRIVE_RUN_DIR / "labeled.json"))
        print(f"✅ 기능1·2 출력 저장: {out_path}")

    # === 라벨 검수 (전체 파일 저장 + 화면 랜덤 20건, seed=42) ===
    _text_by_id = {m.get("message_id"): m.get("text", "") for m in RAW_CHAT_MESSAGES}
    has_text = any(_text_by_id.values())
    EMO = {"positive": "긍정", "negative": "부정", "neutral": "중립"}
    def _emo(k): return EMO.get(k, str(k))
    def _agg(v): return f"공격{v}"
    def _txt(mid): return _text_by_id.get(mid, "") or "(원문 없음 — 라벨만 입력됨)"
    n = len(labeled_messages)
    emo_cnt = Counter(m["emotion_label"] for m in labeled_messages)
    agg_cnt = Counter(m["aggression_label"] for m in labeled_messages)
    csv_path = RUN_DIR / "label_review.csv"
    txt_path = RUN_DIR / "label_review.txt"
    with open(csv_path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["message_id", "timestamp", "sender_id", "text", "emotion", "aggression"])
        for m in labeled_messages:
            w.writerow([m["message_id"], m["timestamp"], m["sender_id"],
                        _txt(m["message_id"]), _emo(m["emotion_label"]), m["aggression_label"]])
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"기능 1·2 라벨 검수 결과 (총 {n}건)\n")
        f.write("=" * 70 + "\n")
        for m in labeled_messages:
            t = str(m["timestamp"])[:16].replace("T", " ")
            f.write(f'[{m["sender_id"]} | {t}] "{_txt(m["message_id"])}"  →  '
                    f'{_emo(m["emotion_label"])} / {_agg(m["aggression_label"])}\n')
    if DRIVE_RUN_DIR:
        for p in (csv_path, txt_path):
            try:
                shutil.copy(str(p), str(DRIVE_RUN_DIR / p.name))
            except Exception:
                pass
    emo_brief = " / ".join(f"{_emo(k)}{emo_cnt[k]}" for k in ["positive", "negative", "neutral"] if emo_cnt.get(k))
    agg_brief = " / ".join(f"{k}:{agg_cnt[k]}" for k in [0, 1, 2] if agg_cnt.get(k))
    if not has_text:
        print("⚠️ 입력에 원문(text)이 없어 라벨만 저장했습니다.")
    print(f"✅ 라벨 검수 저장: label_review.csv / .txt (총 {n}건) | 감정 {emo_brief} · 공격 {agg_brief}")
    random.seed(42)  # 재현성(Seed=42) — 누가 돌려도 같은 20건
    k = min(20, n)
    sample = sorted(random.sample(labeled_messages, k), key=lambda m: m["message_id"])
    print(f"── 라벨 검수 미리보기 (랜덤 {k}건, seed=42) ──")
    for m in sample:
        t = str(m["timestamp"])[:16].replace("T", " ")
        print(f'[{m["sender_id"]} | {t}] "{_txt(m["message_id"])}"  →  '
              f'{_emo(m["emotion_label"])} / {_agg(m["aggression_label"])}')

    # === 기능 3 날짜별 분석 → daily_report.json ===
    daily_reports = run_daily(labeled_messages)
    rep_path = RUN_DIR / "daily_report.json"
    with open(rep_path, "w", encoding="utf-8") as f:
        json.dump(daily_reports, f, ensure_ascii=False, indent=2)
    if DRIVE_RUN_DIR:
        shutil.copy(str(rep_path), str(DRIVE_RUN_DIR / "daily_report.json"))
    print(f"✅ 리포트 생성: {rep_path} (날짜별 {len(daily_reports)}건)")

    # === 시각화 ===
    _render_report(daily_reports)

    # === 자체 점검 (재현성·적법성 증빙) ===
    grouped = group_messages_by_sender(labeled_messages)
    for sender_id, msgs in grouped.items():
        today, baseline = split_time_window(msgs, REFERENCE_DATE)
        print(f"[{sender_id}] today={len(today)}, baseline={len(baseline)}")
    assert MIN_MSG_COUNT == 10 and MIN_BASELINE_MSG_COUNT == 10
    print("가드레일 상수 OK\n")


In [ ]:
# ── 실데이터 카카오톡 로그를 순서대로 모두 실행 (chat_log1 → chat_log2) ──
#   각 로그는 outputs/<입력명>_<시각>/ 폴더에 따로 저장되어 서로 덮어쓰지 않습니다.
INPUTS_TO_RUN = ["chat_log1_raw.json", "chat_log2_raw.json"]

present = [name for name in INPUTS_TO_RUN if (WORK_DIR / name).exists()]
if not present:
    raise FileNotFoundError(
        "실데이터(chat_log1_raw.json·chat_log2_raw.json)가 없습니다. "
        "preprocess_chat_log.py로 전처리하거나 저장소를 다시 clone하세요."
    )

print("실행 대상:", present)
for name in present:
    process_input(name)
print("\n\U0001f389 전체 완료 — 각 로그 결과는 outputs/<입력명>_<시각>/ 폴더 참고")
